# Fuzzy Logic Based Emergency Communication Priority System

**Milestone-2 — Phase 4 (Testing) · Phase 5 (Analysis) · Phase 6 (User Interface)**

| | |
|---|---|
| **Subject** | BE05000461 — Fuzzy Logic & Genetic Programming |
| **Institute** | L. D. College of Engineering, Ahmedabad (GTU) |
| **Inference Method** | Mamdani (AND = MIN, Aggregation = MAX, Defuzzification = Weighted Average) |

---

## Project Introduction

This notebook continues directly from **Phase 2 (System Design)**, which defined the linguistic
variables, membership functions, and the 8-rule knowledge base for a Mamdani fuzzy inference
system that computes a **Call Priority Score (0–100)** for emergency communication calls.

The controller takes three crisp inputs:

- **Emergency Severity** (0 – 10) — *Low, Moderate, Critical*
- **Network Congestion** (0 – 100 %) — *Low, Medium, High*
- **Signal Strength / RSSI** (-100 to -30 dBm) — *Weak, Moderate, Strong*

...and produces one crisp output, the **Call Priority Score** (0 – 100), classified as
*Low / Medium / High Priority*.

No membership function parameters or fuzzy rules are changed from the Phase 2 specification.
This notebook implements the controller from scratch using only `numpy`, `pandas`, and
`matplotlib` — no external fuzzy-logic library is required, so it runs anywhere without extra
installation.

**Contents**

1. Fuzzy controller implementation (membership functions, fuzzification, rule base, Mamdani inference, defuzzification)
2. **Phase 4 — Testing**: membership function plots, 12 test cases (including the required worked example), results table, visualizations
3. **Phase 5 — Analysis**: controller behaviour, strengths, weaknesses, scope for improvement
4. **Phase 6 — User Interface**: a Streamlit UI (controller logic kept in a separate module) plus an optional in-notebook interactive demo


## 0. Setup

### 0.1 Install / Import Required Libraries

Only standard scientific-Python packages are needed. If any are missing, uncomment the
`pip install` line below.

In [ ]:
# Uncomment the line below if numpy / pandas / matplotlib are not already installed
# %pip install numpy pandas matplotlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 100
print("Libraries imported successfully.")


### 0.2 Membership Function Primitives

Two shapes are used throughout this project: **trapezoidal** `[a, b, c, d]` and
**triangular** `[a, b, c]` (a triangular function is a trapezoid with `b == c`).

In [ ]:
def trapmf(x, params):
    """Trapezoidal membership function. params = [a, b, c, d]."""
    a, b, c, d = params
    x = np.asarray(x, dtype=float)
    y = np.zeros_like(x)

    # Rising edge a -> b
    if b > a:
        idx = (x >= a) & (x < b)
        y[idx] = (x[idx] - a) / (b - a)
    # Flat top b -> c
    idx = (x >= b) & (x <= c)
    y[idx] = 1.0
    # Falling edge c -> d
    if d > c:
        idx = (x > c) & (x <= d)
        y[idx] = (d - x[idx]) / (d - c)

    return y


def trimf(x, params):
    """Triangular membership function. params = [a, b, c] (peak at b)."""
    a, b, c = params
    return trapmf(x, [a, b, b, c])


def mf_value(x_scalar, shape, params):
    """Evaluate a single crisp scalar x_scalar against a named membership shape."""
    arr = np.array([x_scalar], dtype=float)
    if shape == "trap":
        return float(trapmf(arr, params)[0])
    elif shape == "tri":
        return float(trimf(arr, params)[0])
    else:
        raise ValueError(f"Unknown shape: {shape}")

print("Membership function primitives defined: trapmf(), trimf(), mf_value()")


### 0.3 Linguistic Variable Definitions (from Phase 2 — unchanged)

In [ ]:
# ---- INPUT 1: Emergency Severity (0 - 10) ----
SEVERITY_RANGE = (0, 10)
SEVERITY_MF = {
    "low":      ("trap", [0, 0, 3, 5]),
    "moderate": ("tri",  [3, 5, 7]),
    "critical": ("trap", [5, 7, 10, 10]),
}

# ---- INPUT 2: Network Congestion (0 - 100 %) ----
CONGESTION_RANGE = (0, 100)
CONGESTION_MF = {
    "low":    ("trap", [0, 0, 30, 50]),
    "medium": ("tri",  [30, 50, 70]),
    "high":   ("trap", [50, 70, 100, 100]),
}

# ---- INPUT 3: Signal Strength / RSSI (-100 to -30 dBm) ----
RSSI_RANGE = (-100, -30)
RSSI_MF = {
    "weak":     ("trap", [-100, -100, -85, -70]),
    "moderate": ("tri",  [-85, -70, -55]),
    "strong":   ("trap", [-70, -55, -30, -30]),
}

# ---- OUTPUT: Call Priority Score (0 - 100) ----
PRIORITY_RANGE = (0, 100)
PRIORITY_MF = {
    "low":    ("trap", [0, 0, 30, 50]),
    "medium": ("tri",  [30, 50, 70]),
    "high":   ("trap", [50, 70, 100, 100]),
}

# Peaks used for weighted-average defuzzification (per Phase 2 / project spec)
OUTPUT_PEAKS = {"low": 20, "medium": 50, "high": 80}

print("Linguistic variables (Severity, Congestion, RSSI, Priority) defined.")


---

## Phase 4 — Testing

### 4.1 Membership Function Graphs

Plots for all three inputs and the output variable, as required.

In [ ]:
def plot_mf_set(ax, x_range, mf_dict, title, xlabel):
    x = np.linspace(x_range[0], x_range[1], 500)
    colors = {0: "#2563eb", 1: "#16a34a", 2: "#dc2626"}
    for i, (label, (shape, params)) in enumerate(mf_dict.items()):
        y = trapmf(x, params) if shape == "trap" else trimf(x, params)
        ax.plot(x, y, label=label.capitalize(), color=colors[i % 3], linewidth=2)
        ax.fill_between(x, y, alpha=0.08, color=colors[i % 3])
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Membership degree")
    ax.set_ylim(-0.05, 1.05)
    ax.legend(loc="upper right", fontsize=9)
    ax.grid(alpha=0.3)


fig, axes = plt.subplots(2, 2, figsize=(13, 9))

plot_mf_set(axes[0, 0], SEVERITY_RANGE, SEVERITY_MF,
            "Emergency Severity", "Severity (0-10)")
plot_mf_set(axes[0, 1], CONGESTION_RANGE, CONGESTION_MF,
            "Network Congestion", "Congestion (%)")
plot_mf_set(axes[1, 0], RSSI_RANGE, RSSI_MF,
            "Signal Strength (RSSI)", "RSSI (dBm)")
plot_mf_set(axes[1, 1], PRIORITY_RANGE, PRIORITY_MF,
            "Call Priority Score (Output)", "Priority Score (0-100)")

plt.tight_layout()
plt.savefig("membership_functions.png", bbox_inches="tight")
plt.show()


### 4.2 The Mamdani Fuzzy Controller

**Pipeline:** fuzzify each crisp input &rarr; evaluate all 8 rules with **MIN** (AND) &rarr;
aggregate each output term's firing strength with **MAX** &rarr; **weighted-average**
defuzzification using the fixed output peaks `Low=20, Medium=50, High=80`.

The 8 rules are implemented exactly as specified — no rules are added, removed, or changed:

| # | Rule |
|---|---|
| R1 | IF Severity is Critical AND Congestion is High THEN Priority is High |
| R2 | IF Severity is Low AND Congestion is High THEN Priority is Low |
| R3 | IF Severity is Critical AND RSSI is Weak THEN Priority is High |
| R4 | IF Severity is Moderate AND Congestion is Medium THEN Priority is Medium |
| R5 | IF Severity is Low AND Congestion is Low THEN Priority is Medium |
| R6 | IF Severity is Critical AND Congestion is Low THEN Priority is High |
| R7 | IF Severity is Moderate AND Congestion is Low AND RSSI is Strong THEN Priority is Medium |
| R8 | IF Severity is Low AND RSSI is Weak THEN Priority is Low |

In [ ]:
def fuzzify(value, mf_dict):
    """Return {linguistic_term: membership_degree} for a crisp value."""
    return {term: mf_value(value, shape, params) for term, (shape, params) in mf_dict.items()}


def evaluate_rules(sev_deg, cong_deg, rssi_deg):
    """Evaluate the 8-rule knowledge base. AND = MIN. Returns a dict of rule -> (strength, output_term)."""
    rules = {
        "R1": (min(sev_deg["critical"], cong_deg["high"]), "high"),
        "R2": (min(sev_deg["low"], cong_deg["high"]), "low"),
        "R3": (min(sev_deg["critical"], rssi_deg["weak"]), "high"),
        "R4": (min(sev_deg["moderate"], cong_deg["medium"]), "medium"),
        "R5": (min(sev_deg["low"], cong_deg["low"]), "medium"),
        "R6": (min(sev_deg["critical"], cong_deg["low"]), "high"),
        "R7": (min(sev_deg["moderate"], cong_deg["low"], rssi_deg["strong"]), "medium"),
        "R8": (min(sev_deg["low"], rssi_deg["weak"]), "low"),
    }
    return rules


def aggregate_and_defuzzify(rules):
    """Aggregate rule strengths per output term with MAX, then weighted-average defuzzify."""
    term_degree = {"low": 0.0, "medium": 0.0, "high": 0.0}
    for _, (strength, term) in rules.items():
        term_degree[term] = max(term_degree[term], strength)  # MAX aggregation

    numerator = sum(term_degree[t] * OUTPUT_PEAKS[t] for t in term_degree)
    denominator = sum(term_degree.values())

    if denominator == 0:
        # No rule fired for this input combination (a known limitation of an 8-rule base
        # that does not cover all 27 possible term combinations - see Phase 5).
        return 50.0, term_degree, True  # fallback: neutral Medium score

    score = numerator / denominator
    return score, term_degree, False


def categorize_priority(score):
    """Classify a crisp priority score using the OUTPUT membership functions (max-membership rule)."""
    degrees = fuzzify(score, PRIORITY_MF)
    label = max(degrees, key=degrees.get)
    return {"low": "Low Priority", "medium": "Medium Priority", "high": "High Priority"}[label]


def fuzzy_priority_controller(severity, congestion, rssi, verbose=False):
    """Full Mamdani pipeline: crisp inputs -> crisp Call Priority Score + category."""
    sev_deg = fuzzify(severity, SEVERITY_MF)
    cong_deg = fuzzify(congestion, CONGESTION_MF)
    rssi_deg = fuzzify(rssi, RSSI_MF)

    rules = evaluate_rules(sev_deg, cong_deg, rssi_deg)
    score, term_degree, fallback_used = aggregate_and_defuzzify(rules)
    category = categorize_priority(score)

    if verbose:
        rule_strengths = {k: round(v[0], 3) for k, v in rules.items()}
        print(f"Severity degrees:  {sev_deg}")
        print(f"Congestion degrees:{cong_deg}")
        print(f"RSSI degrees:      {rssi_deg}")
        print(f"Rule strengths:    {rule_strengths}")
        print(f"Aggregated output: {term_degree}")
        print(f"Fallback used:     {fallback_used}")
        print(f"Priority Score:    {score:.2f}  ->  {category}")

    return round(score, 2), category


print("Mamdani controller ready: fuzzy_priority_controller(severity, congestion, rssi)")


### 4.3 Worked Example (Required Verification)

`Severity = 6.5, Congestion = 65, RSSI = -80` &rarr; expected approximate result **72.5**.

In [ ]:
score, category = fuzzy_priority_controller(6.5, 65, -80, verbose=True)
print(f"\nFinal Result: Priority Score = {score}  |  Category = {category}")
print(f"Expected ~72.5  ->  {'MATCH' if abs(score - 72.5) < 0.5 else 'MISMATCH'}")


### 4.4 Test Cases

12 test combinations covering a variety of Low / Moderate / Critical severity scenarios,
different congestion levels, and different signal conditions — including the required worked
example and the boundary/edge case of maximum severity, congestion, and signal strength.

In [ ]:
test_cases = [
    # (description, severity, congestion, rssi)
    ("Worked example (spec)",                 6.5, 65,  -80),
    ("Low severity, low congestion, strong signal",  1.0, 10,  -40),
    ("Low severity, high congestion, weak signal",   1.0, 85,  -95),
    ("Moderate severity, medium congestion, moderate signal", 5.0, 50, -70),
    ("Critical severity, high congestion, weak signal",  9.0, 90, -95),
    ("Critical severity, low congestion, strong signal", 8.5, 15, -35),
    ("Moderate severity, low congestion, strong signal (R7)", 5.5, 10, -35),
    ("Low severity, weak signal (R8)",          2.0, 40,  -90),
    ("Critical severity, medium congestion, moderate signal", 9.5, 55, -72),
    ("Moderate severity, high congestion, moderate signal",   5.0, 75, -65),
    ("Low severity, moderate congestion, moderate signal",    2.5, 45, -60),
    ("Maximum extreme (edge case)",            10.0, 100, -30),
]

rows = []
for i, (desc, sev, cong, rssi) in enumerate(test_cases, start=1):
    score, category = fuzzy_priority_controller(sev, cong, rssi)
    rows.append({
        "Test #": i,
        "Description": desc,
        "Severity": sev,
        "Congestion (%)": cong,
        "RSSI (dBm)": rssi,
        "Fuzzy Priority Score": score,
        "Priority Category": category,
    })

results_df = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 45)
results_df


### 4.5 Visualizing the Test Results

In [ ]:
category_colors = {"Low Priority": "#16a34a", "Medium Priority": "#f59e0b", "High Priority": "#dc2626"}
bar_colors = results_df["Priority Category"].map(category_colors)

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

# --- Bar chart: priority score per test case ---
axes[0].bar(results_df["Test #"], results_df["Fuzzy Priority Score"], color=bar_colors, edgecolor="black", linewidth=0.5)
axes[0].axhline(30, color="gray", linestyle="--", linewidth=0.8)
axes[0].axhline(50, color="gray", linestyle="--", linewidth=0.8)
axes[0].axhline(70, color="gray", linestyle="--", linewidth=0.8)
axes[0].set_xlabel("Test Case #")
axes[0].set_ylabel("Fuzzy Priority Score (0-100)")
axes[0].set_title("Call Priority Score per Test Case", fontweight="bold")
axes[0].set_xticks(results_df["Test #"])
axes[0].set_ylim(0, 100)
axes[0].grid(axis="y", alpha=0.3)

# --- Scatter: severity vs priority score, sized/colored by congestion ---
sc = axes[1].scatter(results_df["Severity"], results_df["Fuzzy Priority Score"],
                      c=results_df["Congestion (%)"], cmap="coolwarm", s=110,
                      edgecolor="black", linewidth=0.6)
axes[1].set_xlabel("Emergency Severity (0-10)")
axes[1].set_ylabel("Fuzzy Priority Score (0-100)")
axes[1].set_title("Severity vs. Priority Score\n(color = Network Congestion %)", fontweight="bold")
axes[1].grid(alpha=0.3)
cbar = plt.colorbar(sc, ax=axes[1])
cbar.set_label("Congestion (%)")

# Shared legend for priority category colors
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in category_colors.values()]
fig.legend(handles, category_colors.keys(), loc="upper center", ncol=3, bbox_to_anchor=(0.5, 1.04))

plt.tight_layout()
plt.savefig("test_results_visualization.png", bbox_inches="tight")
plt.show()


In [ ]:
# Distribution of priority categories across the 12 test cases
category_counts = results_df["Priority Category"].value_counts().reindex(
    ["Low Priority", "Medium Priority", "High Priority"], fill_value=0
)

plt.figure(figsize=(6, 4.5))
plt.bar(category_counts.index, category_counts.values,
        color=[category_colors[c] for c in category_counts.index], edgecolor="black")
plt.ylabel("Number of Test Cases")
plt.title("Distribution of Priority Categories", fontweight="bold")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("category_distribution.png", bbox_inches="tight")
plt.show()


---

## Phase 5 — Analysis

### 5.1 Controller Behaviour

**How Severity affects Priority.** Emergency Severity is the dominant driver of the output.
Whenever severity is fully or partially *Critical*, Rules R1, R3, and R6 push the aggregated
output strongly toward the *High* priority term, regardless of which specific network condition
triggered the rule. Conversely, a fully *Low* severity call is only ever routed toward *Low* or
*Medium* priority (Rules R2, R5, R8) — the controller never assigns *High* priority to a call it
fuzzifies as purely low-severity, which matches real dispatch intuition that a non-critical
report should not compete with a genuine emergency.

**How Network Congestion affects Priority.** Congestion acts mainly as a modifier rather than a
primary driver. A *Critical* call under *High* congestion (R1) and the same call under *Low*
congestion (R6) both resolve to *High* priority — the rule base intentionally never lets network
load suppress a genuinely critical call. For a *Low*-severity call, however, congestion changes
the outcome: *High* congestion drives it to *Low* priority (R2, so as not to waste scarce
bandwidth on a minor call), while *Low* congestion instead nudges it to *Medium* (R5), since
spare capacity is available and there is little cost in giving a routine call a slightly higher
queue position.

**How RSSI affects Priority.** Signal strength matters chiefly at the two extremes. A weak
signal combined with critical severity (R3) reinforces the case for *High* priority — a poor
link means the call is more likely to drop or fail to reach a dispatcher, so it needs to be
served with urgency while it is still connected. A weak signal combined with low severity (R8)
instead reinforces *Low* priority. A strong signal appears only in R7, where it helps push a
*Moderate*-severity, *Low*-congestion call toward *Medium* priority.

**How multiple inputs interact.** Because the AND operator is MIN, a rule's firing strength is
always capped by its *weakest* supporting input — a call must be reasonably well matched on
*every* antecedent term for that rule to contribute strongly. Because several rules can fire at
once (for example, a severity of 6.5 is simultaneously 25% Moderate and 75% Critical), the
controller blends multiple rule conclusions through MAX aggregation and weighted-average
defuzzification rather than picking a single "winning" rule. This is exactly what produces
scores such as 72.5 for the worked example, which sits between the fixed *Medium* (50) and
*High* (80) output peaks rather than landing on either one exactly.

**Smooth, graded response vs. abrupt thresholds.** Because membership functions overlap (e.g.
Severity is simultaneously part-Low and part-Moderate between values 3 and 5), a one-unit change
in an input — say severity moving from 4.9 to 5.1 — only changes the aggregated degrees by a
small amount, producing a correspondingly small change in the final score. A crisp
threshold-based system would instead jump discontinuously from one hard-coded priority to
another at the same boundary. This graded behaviour is visible in the scatter plot above,
where the priority score rises smoothly with severity instead of forming vertical steps.

### 5.2 Strengths of the Controller

- **Handles uncertainty and imprecision.** Inputs like "fairly urgent" or "moderately weak
  signal" are captured directly as partial membership degrees instead of forcing a premature
  crisp classification.
- **Smooth, graded response.** As discussed above, overlapping membership functions and
  weighted-average defuzzification avoid the abrupt jumps that a fixed-threshold system would
  produce at every boundary.
- **Transparent IF–THEN rules.** All 8 rules are human-readable and auditable — a network
  engineer or dispatch supervisor can read and verify the exact reasoning behind any given
  score without inspecting model weights or opaque code.
- **Multi-input decision making.** The controller genuinely reasons jointly over severity,
  congestion, and signal quality rather than prioritizing on severity alone, matching how a
  real dispatcher weighs several factors at once.
- **Flexibility compared to crisp threshold systems.** Because behaviour is defined by
  membership function shapes and simple rules, it is straightforward to widen or narrow a
  linguistic term, or add a rule, without restructuring the entire decision logic — something
  a nested if-else threshold system cannot do gracefully.

### 5.3 Weaknesses of the Controller

- **Manually chosen membership function parameters.** All the trapezoid/triangle breakpoints
  (e.g. Severity "Critical" starting at 5) were selected by the designer's judgement rather than
  derived from data, which may not reflect real dispatch thresholds.
- **Limited rule base — only 8 of 27 possible term combinations are covered.** This is a real,
  observable gap: Test Case 10 above (Severity = 5.0 fully *Moderate*, Congestion = 75 fully
  *High*) fires **no rule at all**, forcing the implementation to fall back to a neutral score.
  Any input combination that does not resemble R1–R8 is handled only by this fallback, not by
  genuine fuzzy reasoning.
- **Not validated against real-world emergency network data.** The rules and membership
  functions encode plausible expert intuition but have not been checked against logged call
  outcomes, dispatcher decisions, or network telemetry.
- **Performance depends entirely on expert-defined rules.** There is no learning or adaptation;
  if the underlying assumptions about dispatch priorities are wrong, the controller will be
  wrong in exactly the same way every time.
- **Weighted-average defuzzification is a simplification.** Using three fixed output peaks
  (20/50/80) instead of full centroid defuzzification over the aggregated output fuzzy set
  loses some resolution — two different aggregated shapes with the same weighted average would
  produce identical scores even if a full centroid calculation would distinguish them.

### 5.4 Scope for Improvement

- **Adding more rules** to cover currently-unhandled input combinations (such as the
  Moderate-severity / High-congestion gap identified above), reducing reliance on the fallback
  score.
- **Using all, or more, of the 27 possible input-term combinations**, systematically working
  through each Severity × Congestion × RSSI triple rather than the 8 combinations judged most
  important for dispatch.
- **Optimizing membership function parameters** using historical call data instead of manual
  estimation, so breakpoints reflect actual dispatch behaviour.
- **Using Genetic Algorithms (or other metaheuristics) for tuning** both the membership function
  shapes and the rule consequents, searching for parameters that minimize error against a
  labelled dataset of past prioritization decisions.
- **Using real-world datasets** of emergency call logs, network congestion metrics, and RSSI
  readings to validate and calibrate the controller before deployment.
- **Real-time network integration**, feeding live congestion and RSSI telemetry from the
  network infrastructure directly into the controller instead of manual/simulated inputs.
- **Improving the user interface**, for example by adding historical score trends, batch
  evaluation of multiple simultaneous calls, and role-based dashboards for dispatch
  supervisors.

---

## Phase 6 — User Interface

A **Streamlit** UI is provided, with the fuzzy controller logic kept in a separate module
(`fuzzy_controller.py`) so the UI file only handles layout and display — exactly as required.

Running the two cells below **writes both files to disk** in the current working directory.
Instructions to launch the Streamlit app follow after that. An optional lightweight in-notebook
demo (using `ipywidgets`, if available) is also provided at the end for a quick interactive
check without leaving the notebook.

In [ ]:
%%writefile fuzzy_controller.py
"""
fuzzy_controller.py
--------------------
Fuzzy Logic Based Emergency Communication Priority System - Mamdani controller logic.
Kept fully separate from the UI (Phase 6) so it can be imported by the Streamlit app,
or by any other interface, without duplicating the fuzzy-logic implementation.

Inference method: Mamdani | AND = MIN | Aggregation = MAX | Defuzzification = Weighted Average
Rules and membership function parameters match the Phase 2 System Design exactly.
"""

import numpy as np

# ---------------------------------------------------------------------
# Membership function primitives
# ---------------------------------------------------------------------

def trapmf(x, params):
    a, b, c, d = params
    x = np.asarray(x, dtype=float)
    y = np.zeros_like(x)
    if b > a:
        idx = (x >= a) & (x < b)
        y[idx] = (x[idx] - a) / (b - a)
    idx = (x >= b) & (x <= c)
    y[idx] = 1.0
    if d > c:
        idx = (x > c) & (x <= d)
        y[idx] = (d - x[idx]) / (d - c)
    return y


def trimf(x, params):
    a, b, c = params
    return trapmf(x, [a, b, b, c])


def mf_value(x_scalar, shape, params):
    arr = np.array([x_scalar], dtype=float)
    return float(trapmf(arr, params)[0]) if shape == "trap" else float(trimf(arr, params)[0])


# ---------------------------------------------------------------------
# Linguistic variable definitions (unchanged from Phase 2)
# ---------------------------------------------------------------------

SEVERITY_RANGE = (0, 10)
SEVERITY_MF = {
    "low":      ("trap", [0, 0, 3, 5]),
    "moderate": ("tri",  [3, 5, 7]),
    "critical": ("trap", [5, 7, 10, 10]),
}

CONGESTION_RANGE = (0, 100)
CONGESTION_MF = {
    "low":    ("trap", [0, 0, 30, 50]),
    "medium": ("tri",  [30, 50, 70]),
    "high":   ("trap", [50, 70, 100, 100]),
}

RSSI_RANGE = (-100, -30)
RSSI_MF = {
    "weak":     ("trap", [-100, -100, -85, -70]),
    "moderate": ("tri",  [-85, -70, -55]),
    "strong":   ("trap", [-70, -55, -30, -30]),
}

PRIORITY_RANGE = (0, 100)
PRIORITY_MF = {
    "low":    ("trap", [0, 0, 30, 50]),
    "medium": ("tri",  [30, 50, 70]),
    "high":   ("trap", [50, 70, 100, 100]),
}

OUTPUT_PEAKS = {"low": 20, "medium": 50, "high": 80}


# ---------------------------------------------------------------------
# Fuzzification, rule base, inference, defuzzification
# ---------------------------------------------------------------------

def fuzzify(value, mf_dict):
    return {term: mf_value(value, shape, params) for term, (shape, params) in mf_dict.items()}


def evaluate_rules(sev_deg, cong_deg, rssi_deg):
    return {
        "R1": (min(sev_deg["critical"], cong_deg["high"]), "high"),
        "R2": (min(sev_deg["low"], cong_deg["high"]), "low"),
        "R3": (min(sev_deg["critical"], rssi_deg["weak"]), "high"),
        "R4": (min(sev_deg["moderate"], cong_deg["medium"]), "medium"),
        "R5": (min(sev_deg["low"], cong_deg["low"]), "medium"),
        "R6": (min(sev_deg["critical"], cong_deg["low"]), "high"),
        "R7": (min(sev_deg["moderate"], cong_deg["low"], rssi_deg["strong"]), "medium"),
        "R8": (min(sev_deg["low"], rssi_deg["weak"]), "low"),
    }


def aggregate_and_defuzzify(rules):
    term_degree = {"low": 0.0, "medium": 0.0, "high": 0.0}
    for _, (strength, term) in rules.items():
        term_degree[term] = max(term_degree[term], strength)

    numerator = sum(term_degree[t] * OUTPUT_PEAKS[t] for t in term_degree)
    denominator = sum(term_degree.values())

    if denominator == 0:
        return 50.0, term_degree, True

    return numerator / denominator, term_degree, False


def categorize_priority(score):
    degrees = fuzzify(score, PRIORITY_MF)
    label = max(degrees, key=degrees.get)
    return {"low": "Low Priority", "medium": "Medium Priority", "high": "High Priority"}[label]


def fuzzy_priority_controller(severity, congestion, rssi):
    """Main entry point used by the UI. Returns (score, category, term_degree, fallback_used)."""
    sev_deg = fuzzify(severity, SEVERITY_MF)
    cong_deg = fuzzify(congestion, CONGESTION_MF)
    rssi_deg = fuzzify(rssi, RSSI_MF)

    rules = evaluate_rules(sev_deg, cong_deg, rssi_deg)
    score, term_degree, fallback_used = aggregate_and_defuzzify(rules)
    category = categorize_priority(score)

    return round(score, 2), category, term_degree, fallback_used


In [ ]:
%%writefile streamlit_app.py
"""
streamlit_app.py
------------------
Phase 6 - User Interface for the Fuzzy Logic Based Emergency Communication Priority System.
All fuzzy-logic computation lives in fuzzy_controller.py; this file only handles the UI.

Run with:   streamlit run streamlit_app.py
"""

import streamlit as st
from fuzzy_controller import fuzzy_priority_controller

st.set_page_config(page_title="Emergency Priority System", page_icon=":rotating_light:", layout="centered")

st.title(":rotating_light: Fuzzy Logic Based Emergency Communication Priority System")
st.caption("Mamdani Fuzzy Inference  |  AND = MIN  |  Aggregation = MAX  |  Defuzzification = Weighted Average")

st.markdown("---")
st.subheader("Inputs")

col1, col2 = st.columns(2)
with col1:
    severity = st.slider("Emergency Severity (0 - 10)", min_value=0.0, max_value=10.0, value=5.0, step=0.1)
    congestion = st.slider("Network Congestion (0 - 100 %)", min_value=0, max_value=100, value=50, step=1)
with col2:
    rssi = st.slider("Signal Strength / RSSI (-100 to -30 dBm)", min_value=-100, max_value=-30, value=-65, step=1)

st.markdown("---")

if st.button("Calculate Priority", type="primary", use_container_width=True):
    score, category, term_degree, fallback_used = fuzzy_priority_controller(severity, congestion, float(rssi))

    color = {"Low Priority": "green", "Medium Priority": "orange", "High Priority": "red"}[category]

    st.subheader("Result")
    st.markdown(f"### Call Priority Score: **{score:.1f} / 100**")
    st.markdown(f"### Priority Category: :{color}[**{category}**]")

    st.progress(min(int(score), 100) / 100)

    if fallback_used:
        st.warning(
            "No rule in the 8-rule knowledge base fired strongly for this exact input "
            "combination; a neutral fallback score was used. See Phase 5 - Weaknesses."
        )

    if category == "High Priority":
        st.error("This call should be routed with the highest dispatch urgency.")
    elif category == "Medium Priority":
        st.warning("This call should be queued with standard/elevated urgency.")
    else:
        st.success("This call can be handled with routine, non-urgent priority.")

    with st.expander("Show aggregated output-term degrees (debug view)"):
        st.json(term_degree)

st.markdown("---")
st.caption("Fuzzy Logic Based Emergency Communication Priority System | BE05000461 - Fuzzy Logic & Genetic Programming | L. D. College of Engineering (GTU)")


### 6.1 Running the Streamlit UI

After running the two cells above (which write `fuzzy_controller.py` and `streamlit_app.py` to
the current working directory), launch the app from a terminal:

```bash
pip install streamlit          # only needed once
streamlit run streamlit_app.py
```

This opens the UI in a browser tab (usually at `http://localhost:8501`) with:

- Sliders for **Emergency Severity**, **Network Congestion**, and **RSSI**
- A **Calculate Priority** button
- The resulting **Call Priority Score**, **Priority Category**, and a colored progress-bar gauge

> **Note:** Streamlit apps run as standalone scripts outside the Jupyter kernel and cannot be
> executed directly inside a notebook cell. The optional demo below provides an equivalent
> interactive experience without leaving the notebook, for quick verification.

### 6.2 Optional: Quick In-Notebook Interactive Demo

Uses `ipywidgets` if it is available in this environment; otherwise it prints instructions
instead of raising an error, so this cell never blocks the rest of the notebook.

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display, HTML

    sev_slider = widgets.FloatSlider(value=5.0, min=0, max=10, step=0.1, description="Severity:")
    cong_slider = widgets.FloatSlider(value=50, min=0, max=100, step=1, description="Congestion (%):")
    rssi_slider = widgets.FloatSlider(value=-65, min=-100, max=-30, step=1, description="RSSI (dBm):")
    button = widgets.Button(description="Calculate Priority", button_style="primary")
    output = widgets.Output()

    def on_click(_):
        with output:
            output.clear_output()
            score, category = fuzzy_priority_controller(sev_slider.value, cong_slider.value, rssi_slider.value)
            color = {"Low Priority": "#16a34a", "Medium Priority": "#f59e0b", "High Priority": "#dc2626"}[category]
            bar_width = min(max(score, 0), 100)
            display(HTML(f"""
            <div style="font-family:sans-serif; padding:10px;">
              <h3>Call Priority Score: {score:.1f} / 100</h3>
              <h4 style="color:{color};">Priority Category: {category}</h4>
              <div style="background:#e5e7eb; border-radius:6px; width:100%; height:22px;">
                <div style="background:{color}; width:{bar_width}%; height:100%; border-radius:6px;"></div>
              </div>
            </div>
            """))

    button.on_click(on_click)
    display(widgets.VBox([sev_slider, cong_slider, rssi_slider, button, output]))

except ImportError:
    print("ipywidgets is not installed in this environment.")
    print("Install it with:  pip install ipywidgets")
    print("Or simply use the Streamlit UI described in section 6.1 above.")


---

## Summary

This notebook implemented and tested the Mamdani fuzzy controller designed in Phase 2, verified
it against the required worked example (Severity=6.5, Congestion=65, RSSI=-80 &rarr; 72.5),
evaluated it on 12 varied test cases (Phase 4), analyzed its behaviour, strengths, weaknesses,
and scope for improvement (Phase 5), and delivered a Streamlit-based interactive UI with the
controller logic kept in a separate module (Phase 6).